<h4>Getting Data<h4>

In [1]:
import numpy as np

In [2]:
def read_all_images(path_to_data):
    """
    :param path_to_data: the file containing the binary images from the STL-10 dataset
    :return: an array containing all the images
    """

    with open(path_to_data, 'rb') as f:
        # read whole file in uint8 chunks
        everything = np.fromfile(f, dtype=np.uint8)

        # We force the data into 3x96x96 chunks, since the
        # images are stored in "column-major order", meaning
        # that "the first 96*96 values are the red channel,
        # the next 96*96 are green, and the last are blue."
        # The -1 is since the size of the pictures depends
        # on the input file, and this way numpy determines
        # the size on its own.

        images = np.reshape(everything, (-1, 3, 96, 96))

        # Now transpose the images into a standard image format
        # readable by, for example, matplotlib.imshow
        # You might want to comment this line or reverse the shuffle
        # if you will use a learning algorithm like CNN, since they like
        # their channels separated.
        images = np.transpose(images, (0, 3, 2, 1))
        return images
    

def read_labels(path_to_labels):
    """
    :param path_to_labels: path to the binary file containing labels from the STL-10 dataset
    :return: an array containing the labels
    """
    with open(path_to_labels, 'rb') as f:
        labels = np.fromfile(f, dtype=np.uint8)
        return labels

In [3]:
DATA_TRAIN_PATH = './data/stl10_binary/train_X.bin'
DATA_TEST_PATH = './data/stl10_binary/test_X.bin'

LABEL_TRAIN_PATH = './data/stl10_binary/train_y.bin'
LABEL_TEST_PATH = './data/stl10_binary/test_y.bin'

In [4]:
X_train = read_all_images(DATA_TRAIN_PATH)
X_train.shape

(5000, 96, 96, 3)

In [5]:
y_train = read_labels(LABEL_TRAIN_PATH)
y_train = y_train - 1
y_train.shape

(5000,)

In [6]:
X_test = read_all_images(DATA_TEST_PATH)
X_test.shape

(8000, 96, 96, 3)

In [7]:
y_test = read_labels(LABEL_TEST_PATH)
y_test.shape

(8000,)

<h2>Pipeline<h2>

In [8]:
import torch
from torchsummary import summary
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import TensorDataset, DataLoader


In [9]:
import torchvision.models as models

pretrained_model = models.resnet50(weights="IMAGENET1K_V1")

In [10]:
class ModifiedResNet(nn.Module):
    def __init__(self):
        super(ModifiedResNet, self).__init__()
        self.resnet = torch.hub.load('pytorch/vision', 'resnet50', weights="IMAGENET1K_V1")
        num_classes = 10  # STL-10 classes
        self.resnet.fc = nn.Linear(pretrained_model.fc.in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

model = ModifiedResNet()


Using cache found in C:\Users\Nazarii/.cache\torch\hub\pytorch_vision_main


In [11]:
for param in model.parameters():
    param.requires_grad = False


In [12]:
image_size = 96

mean = torch.tensor([0.05438065, 0.05291743, 0.07920227])
std = torch.tensor([0.39414383, 0.33547948, 0.38544176])

transform_train = T.Compose([
    T.Resize(image_size + 4),
    T.CenterCrop(image_size),
    T.RandomRotation(20),
    T.RandomAffine(
        degrees=10,
        translate=(0.01, 0.12),
        shear=(0.01, 0.03)
    ),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ToTensor(),
    T.Normalize(mean, std, inplace=True),
    T.RandomErasing(inplace=True),
])

transform_val = T.Compose([
    T.Resize(image_size),
    T.ToTensor(),
    T.Normalize(mean, std, inplace=True),
])


In [13]:
train_dataset = TensorDataset(torch.Tensor(X_train).permute(0, 3, 1, 2), torch.Tensor(y_train).long())
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [14]:
class CustomResNet(nn.Module):
    def __init__(self, pretrained_model):
        super(CustomResNet, self).__init__()
        self.resnet = torch.hub.load('pytorch/vision', 'resnet50', weights="IMAGENET1K_V1")
        self.features = nn.Sequential(*list(pretrained_model.children())[:-1])
        self.classifier = nn.Linear(pretrained_model.fc.in_features, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = CustomResNet(pretrained_model)

Using cache found in C:\Users\Nazarii/.cache\torch\hub\pytorch_vision_main


In [15]:
default_criterion = nn.CrossEntropyLoss()
default_optimizer = optim.Adam(model.parameters(), lr=0.001)
summary(model, (3, 96, 96))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 48, 48]           9,408
       BatchNorm2d-2           [-1, 64, 48, 48]             128
              ReLU-3           [-1, 64, 48, 48]               0
         MaxPool2d-4           [-1, 64, 24, 24]               0
            Conv2d-5           [-1, 64, 24, 24]           4,096
       BatchNorm2d-6           [-1, 64, 24, 24]             128
              ReLU-7           [-1, 64, 24, 24]               0
            Conv2d-8           [-1, 64, 24, 24]          36,864
       BatchNorm2d-9           [-1, 64, 24, 24]             128
             ReLU-10           [-1, 64, 24, 24]               0
           Conv2d-11          [-1, 256, 24, 24]          16,384
      BatchNorm2d-12          [-1, 256, 24, 24]             512
           Conv2d-13          [-1, 256, 24, 24]          16,384
      BatchNorm2d-14          [-1, 256,

In [16]:
for param in model.resnet.layer4.parameters():
    param.requires_grad = True

model.train()

CustomResNet(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
      

In [17]:
val_dataset = TensorDataset(torch.Tensor(X_test).permute(0, 3, 1, 2), torch.Tensor(y_test).long())
val_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [18]:
def trainer(own_model, num_epochs, criterion, optimizer, device):
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                own_model.train()  # Set model to training mode
            else:
                own_model.eval()   # Set model to evaluation mode

            running_loss = 0.0
            running_corrects = 0
            total_samples = 0

            # Iterate over data.
            if phase == 'train':
                dataloader = train_loader
            else:
                dataloader = val_loader

            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = own_model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass and optimization in the training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data).item()
                total_samples += labels.size(0)

            epoch_loss = running_loss / total_samples
            epoch_acc = running_corrects / total_samples

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Deep copy the model if validation accuracy improves
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = own_model.state_dict()

        print()

    # Load the best model weights
    own_model.load_state_dict(best_model_wts)

    print(f'Best val Acc: {best_acc:.4f}')


In [20]:
default_num_epochs = 10
default_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
best_model_wts = model.state_dict()

trainer(own_model=model, num_epochs=default_num_epochs, device=default_device, optimizer=default_optimizer, criterion=default_criterion)
# Time: 54m 10.9s

Epoch 1/10
----------
train Loss: 1.1000 Acc: 0.6316
val Loss: 1.3557 Acc: 0.5988

Epoch 2/10
----------
train Loss: 0.6710 Acc: 0.7796
val Loss: 0.6964 Acc: 0.7478

Epoch 3/10
----------
train Loss: 0.4651 Acc: 0.8406
val Loss: 1.5765 Acc: 0.6010

Epoch 4/10
----------
train Loss: 0.3308 Acc: 0.8936
val Loss: 0.2882 Acc: 0.9060

Epoch 5/10
----------
train Loss: 0.2965 Acc: 0.9064
val Loss: 0.3603 Acc: 0.8748

Epoch 6/10
----------
train Loss: 0.4268 Acc: 0.8606
val Loss: 0.2962 Acc: 0.9024

Epoch 7/10
----------
train Loss: 0.2571 Acc: 0.9146
val Loss: 0.1854 Acc: 0.9394

Epoch 8/10
----------
train Loss: 0.1048 Acc: 0.9670
val Loss: 0.1460 Acc: 0.9500

Epoch 9/10
----------
train Loss: 0.1221 Acc: 0.9614
val Loss: 0.1148 Acc: 0.9606

Epoch 10/10
----------
train Loss: 0.1332 Acc: 0.9570
val Loss: 0.0872 Acc: 0.9702

Best val Acc: 0.9702


<h2>Bonus task<h2>

In [21]:
class CustomBonusResNet(nn.Module):
    def __init__(self):
        super(CustomBonusResNet, self).__init__()

        self.resnet = models.resnet50(weights=None)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 10)

    def forward(self, x):
        return self.resnet(x)

In [22]:
bonus_num_epochs = 10
bonus_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

bonus_model = CustomBonusResNet().to(bonus_device)
bonus_criterion = nn.CrossEntropyLoss()
bonus_optimizer = optim.Adam(bonus_model.parameters(), lr=0.001)

In [23]:
trainer(own_model=bonus_model, num_epochs=bonus_num_epochs, device=bonus_device, optimizer=bonus_optimizer, criterion=bonus_criterion)
# Time: 46m 21.4s

Epoch 1/10
----------
train Loss: 2.0911 Acc: 0.2728
val Loss: 2.8609 Acc: 0.2270

Epoch 2/10
----------
train Loss: 1.5963 Acc: 0.3932
val Loss: 2.0476 Acc: 0.4112

Epoch 3/10
----------
train Loss: 1.4635 Acc: 0.4606
val Loss: 1.6835 Acc: 0.4628

Epoch 4/10
----------
train Loss: 1.2641 Acc: 0.5296
val Loss: 1.4639 Acc: 0.4682

Epoch 5/10
----------
train Loss: 1.1011 Acc: 0.5966
val Loss: 1.0530 Acc: 0.6206

Epoch 6/10
----------
train Loss: 1.0013 Acc: 0.6430
val Loss: 1.3704 Acc: 0.5488

Epoch 7/10
----------
train Loss: 0.8339 Acc: 0.6976
val Loss: 1.1196 Acc: 0.6292

Epoch 8/10
----------
train Loss: 0.7602 Acc: 0.7232
val Loss: 0.9186 Acc: 0.6822

Epoch 9/10
----------
train Loss: 0.6730 Acc: 0.7568
val Loss: 0.8795 Acc: 0.6842

Epoch 10/10
----------
train Loss: 0.5507 Acc: 0.8010
val Loss: 1.4289 Acc: 0.5956

Best val Acc: 0.6842


<h2>Compare<h2>

When training only the top layers with pre-trained weights, accuracy improved significantly faster, reaching a high validation accuracy of 97.02% by epoch 10. In contrast, training from scratch reached a best validation accuracy of only 68.42%, indicating that using pre-trained weights helps the model converge faster and achieve higher accuracy within a limited number of epochs. In the pre-trained model, the validation loss consistently decreased, reaching much lower values, while in the scratch model, validation loss fluctuated and remained relatively high, suggesting that pre-trained models better generalize to validation data. \
In summary, fine-tuning a pre-trained model proved more effective in terms of both accuracy and generalization than training from scratch, especially given the relatively small size of the STL-10 dataset.

<h1>Theoretical assignment<h1>


2. <strong>Take a look at this paper making a study on the difference between CNN and ViT models.</strong> \
<strong>Answer:</strong> This study explores the representational differences between Convolutional Neural Networks (CNNs) and Vision Transformers (ViTs), focusing on how their distinct architectures affect learning.
3. <strong>What are the questions being studied?</strong> \
<strong>Answer:</strong>
- Representation Similarity and Center Kernel Alignment
- Spartial localization (mostly for ViTs)
- Information (both local and global) in layer representations of both approaches
- Transfer Learning experiments
- Linear probes

4. <strong>What are the conclusions of the paper?</strong> \
<strong>Answer:</strong> ViTs learn more globally-aware representations than CNNs, thanks to their global self-attention mechanisms. Skip connections play a crucial role in ViTs, propagating information from earlier layers to deeper ones. ViTs, especially those using CLS tokens, demonstrate stronger spatial localization capabilities than CNNs. Larger ViTs pretrained on massive datasets develop significantly stronger representations than CNNs.